# 02 응용: Visibility sample에서 재방문 주기 계산

연속된 coverage sample을 하나의 access event로 합치고, 이전 event 종료부터 다음 event 시작까지의 gap을 계산한다.

In [1]:
from statistics import mean

def access_intervals(times_s, visible):
    if len(times_s) != len(visible):
        raise ValueError('시간과 visibility 길이가 다릅니다')
    intervals=[]; start=None; previous=None
    for t,on in zip(times_s,visible):
        if on and start is None: start=t
        if not on and start is not None:
            intervals.append((start,previous)); start=None
        previous=t
    if start is not None: intervals.append((start,previous))
    return intervals

def revisit_gaps(intervals):
    return [b[0]-a[1] for a,b in zip(intervals,intervals[1:])]

times = list(range(0,121,10))
visible = [0,1,1,0,0,1,1,1,0,0,1,0,0]
events = access_intervals(times,visible)
gaps = revisit_gaps(events)
events, gaps

([(10, 20), (50, 70), (100, 100)], [30, 30])

In [2]:
def summarize(gaps_s):
    if not gaps_s: return {'count':0,'min_s':None,'average_s':None,'max_s':None}
    return {'count':len(gaps_s),'min_s':min(gaps_s),
            'average_s':mean(gaps_s),'max_s':max(gaps_s)}

summary = summarize(gaps)
assert events == [(10,20),(50,70),(100,100)]
assert gaps == [30,30]
summary

{'count': 2, 'min_s': 30, 'average_s': 30, 'max_s': 30}

첫 event 이전과 마지막 event 이후의 공백은 관측 구간 경계에서 잘린 censored gap이다. 이를 일반 gap처럼 넣으면 maximum과 average가 simulation duration에 따라 왜곡된다. 논문 재현 시 포함 규칙을 명시해야 한다.

In [3]:
# 논문 Table 3 수치를 다시 집계하고 평균만으로 설계를 고르는 위험을 본다.
five = [45.7,43.7,45.3,43.9,43.7]
six = [45.4,43.5,46.3,43.2,45.2,44.6]
print('5면 phase 평균:',round(mean(five),2),'min')
print('6면 phase 평균:',round(mean(six),2),'min')
print('논문 histogram maximum band: 5면 180~200, 6면 140~160 min')

5면 phase 평균: 44.46 min
6면 phase 평균: 44.7 min
논문 histogram maximum band: 5면 180~200, 6면 140~160 min


In [4]:
# 시간 해상도 오차: event boundary는 sample 간격만큼 불확실하다.
for step_s in (10,30,60,120):
    two_boundary_error_s = 2*step_s
    print(f'{step_s:3d}s step -> gap 최악 경계오차 약 ±{two_boundary_error_s}s 범위 검토')

 10s step -> gap 최악 경계오차 약 ±20s 범위 검토
 30s step -> gap 최악 경계오차 약 ±60s 범위 검토
 60s step -> gap 최악 경계오차 약 ±120s 범위 검토
120s step -> gap 최악 경계오차 약 ±240s 범위 검토


## 실무 체크

관측 시작과 종료를 interpolation했는가, 같은 pass의 짧은 false sample을 합칠 것인가, AOI 전체 coverage와 any-point coverage 중 무엇인가, 관측 기간 경계의 gap을 제외했는가를 결과와 함께 기록한다.